# Testando duas ferramentas de AutoML com o Iris

Nesta atividade, vou comparar o **FLAML** e o **Auto-Sklearn** usando os mesmos dados. Quero verificar qual acerta mais, quanto tempo cada um leva e como ficam os modelos encontrados.

Para a comparação ser justa, os dois terão **60 segundos de busca** e a divisão dos dados será a mesma.

## 1. Dataset e separação dos dados

In [1]:
from pathlib import Path
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

local = Path('/content') if Path('/content').exists() else Path.cwd()
base = load_iris()

x_train, x_test, y_train, y_test = train_test_split(
    base.data, base.target, test_size=0.20,
    random_state=42, stratify=base.target
)

# Guardo a divisão para os dois programas usarem os mesmos exemplos.
np.savez(local / 'dados_iris.npz',
         x_train=x_train, x_test=x_test,
         y_train=y_train, y_test=y_test)

print('Quantidade de flores:', len(base.data))
print('Treino:', len(x_train), '| Teste:', len(x_test))
print('Classes:', ', '.join(base.target_names))

Quantidade de flores: 150
Treino: 120 | Teste: 30
Classes: setosa, versicolor, virginica


## 2. Bibliotecas

Como as duas ferramentas não trabalham bem com as mesmas versões de Python no Colab, o código instala cada uma separadamente. Essa parte é só para conseguir executar a comparação.

In [2]:
import subprocess

executavel = local / 'bin' / 'micromamba'
if not executavel.exists():
    subprocess.run(
        'curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest '
        f'| tar -xj -C {local} bin/micromamba',
        shell=True, executable='/bin/bash', check=True
    )

def criar_ambiente(nome, pacotes):
    diretorio = local / nome
    if not (diretorio / 'bin/python').exists():
        subprocess.run([
            str(executavel), 'create', '-y', '-p', str(diretorio),
            '-c', 'conda-forge', *pacotes
        ], check=True)
    return diretorio / 'bin/python'

python_flaml = criar_ambiente('amb_flaml_alternativo', [
    'python=3.11', 'pip', 'numpy=1.26.4',
    'scipy=1.13.1', 'scikit-learn=1.4.2'
])
subprocess.run([
    str(python_flaml), '-m', 'pip', 'install', '-q',
    'flaml[automl]==2.3.6', 'lightgbm==4.6.0'
], check=True)

python_autosklearn = criar_ambiente('amb_autosklearn_alternativo', [
    'python=3.9', 'auto-sklearn=0.15.0'
])
print('Instalação concluída.')

Instalação concluída.


## 3. Treinamento e avaliação

Este teste usa acurácia e classification_report. Também vou guardar o tempo de treinamento e informações sobre os modelos escolhidos.

In [3]:
# O mesmo conjunto de dados é usado nos dois testes.
programa_teste = r'''import json
import sys
import time
from pathlib import Path
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

caminho = Path(sys.argv[1])
opcao = sys.argv[2]
matriz = np.load(caminho / 'dados_iris.npz')
x_train = matriz['x_train']
x_test = matriz['x_test']
y_train = matriz['y_train']
y_test = matriz['y_test']

inicio = time.perf_counter()

if opcao == 'flaml':
    from flaml import AutoML
    from flaml.automl.data import get_output_from_log

    busca = AutoML()
    arquivo_log = str(caminho / 'historico_flaml.log')
    busca.fit(
        X_train=x_train, y_train=y_train,
        task='classification', metric='accuracy',
        time_budget=60, seed=42, n_jobs=1,
        estimator_list=['lgbm', 'rf', 'extra_tree'],
        eval_method='holdout', split_ratio=0.25,
        log_file_name=arquivo_log, log_type='all', verbose=0
    )
    duracao = time.perf_counter() - inicio
    resposta = busca.predict(x_test)
    try:
        entradas, *_ = get_output_from_log(arquivo_log, time_budget=360)
        avaliacoes = len(entradas)
    except Exception:
        avaliacoes = None
    descricao = f'{busca.best_estimator}: {busca.best_config}'
    quantidade = 1

elif opcao == 'autosklearn':
    from autosklearn.classification import AutoSklearnClassifier
    from autosklearn.metrics import accuracy

    busca = AutoSklearnClassifier(
        time_left_for_this_task=60,
        per_run_time_limit=15,
        memory_limit=2048,
        metric=accuracy,
        seed=42,
        n_jobs=1,
        initial_configurations_via_metalearning=0,
        resampling_strategy='holdout',
        resampling_strategy_arguments={'train_size': 0.75},
        ensemble_size=20
    )
    busca.fit(x_train, y_train, dataset_name='iris')
    duracao = time.perf_counter() - inicio
    resposta = busca.predict(x_test)
    integrantes = busca.get_models_with_weights()
    quantidade = len(integrantes)
    avaliacoes = len(busca.cv_results_['mean_test_score'])
    descricao = '\n'.join(
        f'Peso {peso:.2f} | {str(modelo)[:350]}'
        for peso, modelo in integrantes
    )
else:
    raise ValueError('Ferramenta não reconhecida')

resultado = {
    'acuracia': float(accuracy_score(y_test, resposta)),
    'segundos': round(duracao, 2),
    'avaliacoes': avaliacoes,
    'descricao': descricao,
    'qtd_modelos': quantidade,
    'relatorio': classification_report(
        y_test, resposta,
        target_names=['setosa', 'versicolor', 'virginica'],
        zero_division=0
    )
}
(caminho / f'avaliacao_{opcao}.json').write_text(
    json.dumps(resultado, ensure_ascii=False), encoding='utf-8'
)
'''
(local / 'rodar_auto_ml.py').write_text(programa_teste, encoding='utf-8')
print('Testes preparados.')

Testes preparados.


## 4. Resultados da comparação

In [4]:
import json
import time
import psutil
import pandas as pd
from IPython.display import display

# Executa um teste e acompanha aproximadamente o uso máximo de RAM.
def executar(python, nome):
    saida = local / f'avaliacao_{nome}.json'
    saida.unlink(missing_ok=True)

    comando = [str(python), str(local / 'rodar_auto_ml.py'), str(local), nome]
    inicio = time.perf_counter()
    ram_maxima = 0

    with (local / f'log_{nome}.txt').open('w') as log:
        processo = subprocess.Popen(comando, stdout=log, stderr=subprocess.STDOUT)
        while processo.poll() is None:
            try:
                principal = psutil.Process(processo.pid)
                lista = [principal, *principal.children(recursive=True)]
                ram_agora = sum(p.memory_info().rss for p in lista if p.is_running())
                ram_maxima = max(ram_maxima, ram_agora)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
            time.sleep(0.2)

    if processo.returncode != 0 or not saida.exists():
        print((local / f'log_{nome}.txt').read_text(errors='replace')[-3000:])
        raise RuntimeError(f'O teste de {nome} não terminou corretamente.')

    dados = json.loads(saida.read_text(encoding='utf-8'))
    dados['total'] = round(time.perf_counter() - inicio, 2)
    dados['ram_mb'] = round(ram_maxima / (1024**2), 1)
    return dados

r_flaml = executar(python_flaml, 'flaml')
r_auto = executar(python_autosklearn, 'autosklearn')

linhas = []
for nome, valores in [('FLAML', r_flaml), ('Auto-Sklearn', r_auto)]:
    linhas.append({
        'Ferramenta': nome,
        'Acurácia': f"{valores['acuracia']:.2%}",
        'Treinamento (s)': valores['segundos'],
        'Tempo total (s)': valores['total'],
        'RAM máxima aprox. (MB)': valores['ram_mb'],
        'Registros de busca': valores['avaliacoes'],
        'Modelos finais': valores['qtd_modelos']
    })

display(pd.DataFrame(linhas))

for nome, valores in [('FLAML', r_flaml), ('Auto-Sklearn', r_auto)]:
    print(f'\n--- {nome} ---')
    print(valores['relatorio'])
    print('Modelo(s) encontrado(s):')
    print(valores['descricao'])

if r_flaml['acuracia'] == r_auto['acuracia']:
    print('\nAs duas ferramentas empataram na acurácia.')
else:
    vencedor = 'FLAML' if r_flaml['acuracia'] > r_auto['acuracia'] else 'Auto-Sklearn'
    print(f'\nMaior acurácia nesta execução: {vencedor}.')

,Ferramenta,Acurácia,Treinamento (s),Tempo total (s),RAM máxima aprox. (MB),Registros de busca,Modelos finais
0,FLAML,90.00%,60.65,62.25,177.0,2030,1
1,Auto-Sklearn,96.67%,55.03,56.77,478.4,15,7



--- FLAML ---
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30

Modelo(s) encontrado(s):
rf: {'n_estimators': 27, 'max_leaves': 32, 'max_features': np.float64(0.267847370832647), 'criterion': np.str_('entropy')}

--- Auto-Sklearn ---
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

Modelo(s) encontrado(s):
Peso 0.20 | SimpleClassifica

## 5. Conclusão

Nesta atividade, comparei o desempenho do FLAML e do Auto-Sklearn utilizando o conjunto de dados Iris. O FLAML alcançou uma acurácia de **90%**, enquanto o Auto-Sklearn obteve **96,67%**. Com isso, não houve empate, e o Auto-Sklearn apresentou o melhor resultado, acertando 29 das 30 amostras utilizadas no teste, enquanto o FLAML acertou 27.

Em relação ao custo computacional, o FLAML levou **62,25 segundos** no total e utilizou aproximadamente **177 MB de memória RAM**. Já o Auto-Sklearn terminou em **56,77 segundos**, mas consumiu cerca de **478,4 MB de RAM**. O FLAML apresentou 2.030 registros de busca, enquanto o Auto-Sklearn apresentou apenas 15. Assim, foi possível perceber que o Auto-Sklearn teve um tempo de execução menor, enquanto o FLAML utilizou menos memória e apresentou uma quantidade maior de registros durante a busca.

Quanto à análise dos modelos, o **FLAML foi mais simples de entender**, pois apresentou apenas um modelo final, o Random Forest, com seus parâmetros. O Auto-Sklearn, por outro lado, gerou um conjunto de sete modelos, tornando a análise mais detalhada e um pouco mais complexa. Portanto, considero que o Auto-Sklearn se destacou pela acurácia e pelo tempo de execução, enquanto o FLAML apresentou vantagens no consumo de memória e na facilidade de interpretar o modelo escolhido.